<a href="https://colab.research.google.com/github/philippenchev98/ab-testing-product-analysis/blob/main/AB_Testing_Product_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


#ГЕНЕРИРАНЕ НА РЕАЛИСТИЧНИ ДАННИ ЗА A/B ТЕСТ
np.random.seed(42) #Гарантира, че всеки път ще получаваме едни и същи "случайни" числа

#Група A (Контролна група - Стар дизайн)
n_A = 1000
#Симулираме, че 10% от хората със стария дизайн са извършили покупка
conversions_A = np.random.binomial(1, 0.10, n_A)

#Група B (Експериментална група - Нов дизайн)
n_B = 1000
#Симулираме, че 13% от хората с новия дизайн са извършили покупка, т.е. има 3% подобрение
conversions_B = np.random.binomial(1, 0.13, n_B)

#Създаваме Pandas DataFrame, за да работим лесно с данните
df_A = pd.DataFrame({"User_ID": range(1, n_A + 1), "Group": "A (Old Design)", "Converted": conversions_A})
df_B = pd.DataFrame({"User_ID": range(n_A + 1, n_A + n_B + 1), "Group": "B (New Design)", "Converted": conversions_B})

df = pd.concat([df_A, df_B])

print(f"Общ брой потребители в теста: {df.shape[0]}")
display(df.sample(5)) #Показваме 5 случайни реда

Общ брой потребители в теста: 2000


,User_ID,Group,Converted
537,538,A (Old Design),0
443,444,A (Old Design),0
33,34,A (Old Design),1
668,669,A (Old Design),0
316,1317,B (New Design),0


In [ ]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
import math

#ИЗЧИСЛЯВАНЕ НА CONVERSION RATE (CR)
#Групираме данните, за да видим колко хора са извършили покупка във всяка група
ab_summary = df.groupby("Group")["Converted"].agg(["count", "sum", "mean"])
ab_summary.columns = ["Total_Users", "Purchases", "Conversion_Rate"]

#Запазваме суровите стойности за теста по-долу (Индекс 0 е Група A, Индекс 1 е Група B)
successes = ab_summary["Purchases"].values
nobs = ab_summary["Total_Users"].values

#Форматираме Conversion Rate като процент
ab_summary["Conversion_Rate_Formatted"] = (ab_summary["Conversion_Rate"] * 100).round(2).astype(str) + "%"
display(ab_summary[["Total_Users", "Purchases", "Conversion_Rate_Formatted"]])


#СТАТИСТИЧЕСКИ ТЕСТ (Z-Test за две пропорции)
#Използваме statsmodels, за да изчислим p-value
z_stat, p_value = proportions_ztest(successes, nobs)

print("\nРЕЗУЛТАТИ ОТ СТАТИСТИЧЕСКИЯ ТЕСТ")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")


#ДОВЕРИТЕЛЕН ИНТЕРВАЛ (Confidence Interval)
p_A = successes[0] / nobs[0]
p_B = successes[1] / nobs[1]
q_A = 1 - p_A
q_B = 1 - p_B

#Точкова оценка (Point Estimate) за подобрението (B - A)
point_estimate = p_B - p_A

#Стандартна грешка (SE - Unpooled)
se = math.sqrt((p_B * q_B / nobs[1]) + (p_A * q_A / nobs[0]))

#95% Margin of error (Z_alpha/2 = 1.96 за 95% доверителност)
margin_of_error = 1.96 * se

#Граници на доверителния интервал
ci_lower = point_estimate - margin_of_error
ci_upper = point_estimate + margin_of_error

print("\n95% ДОВЕРИТЕЛЕН ИНТЕРВАЛ ЗА ПОДОБРЕНИЕТО (p_B - p_A)")
print(f"Точкова оценка на разликата: +{point_estimate*100:.2f}%")
print(f"Стандартна грешка (SE): {se:.4f}")
print(f"Margin of error: ±{margin_of_error*100:.2f}%")
print(f"95% CI: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")


print("\nБИЗНЕС ЗАКЛЮЧЕНИЕ")
if p_value < 0.05:
    print("РЕЗУЛТАТ: P-value е под 0.05! Разликата е СТАТИСТИЧЕСКИ ЗНАЧИМА.")
    print("Новият дизайн (Група B) категорично печели и носи повече продажби.")
    print(f"Очакваме реалното увеличение на конверсиите да бъде между {ci_lower*100:.2f}% и {ci_upper*100:.2f}%.")
    print("Можем да го пуснем към всички потребители!")
else:
    print("РЕЗУЛТАТ: P-value е над 0.05. Разликата НЕ Е статистически значима.")
    print("Промяната може да е просто плод на случайност. Не се препоръчва пускане на новия дизайн/Изискват се допълнителни данни.")

,Total_Users,Purchases,Conversion_Rate_Formatted
Group,,,
A (Old Design),1000,100,10.0%
B (New Design),1000,131,13.1%



РЕЗУЛТАТИ ОТ СТАТИСТИЧЕСКИЯ ТЕСТ
Z-statistic: -2.1687
P-value: 0.0301

95% ДОВЕРИТЕЛЕН ИНТЕРВАЛ ЗА ПОДОБРЕНИЕТО (p_B - p_A)
Точкова оценка на разликата: +3.10%
Стандартна грешка (SE): 0.0143
Margin of error: ±2.80%
95% CI: [0.30%, 5.90%]

БИЗНЕС ЗАКЛЮЧЕНИЕ
РЕЗУЛТАТ: P-value е под 0.05! Разликата е СТАТИСТИЧЕСКИ ЗНАЧИМА.
Новият дизайн (Група B) категорично печели и носи повече продажби.
Очакваме реалното увеличение на конверсиите да бъде между 0.30% и 5.90%.
Можем да го пуснем към всички потребители!
